# ML-02 — Research Question and Provisional Lane

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I worked this from the lane guide (`docs/ml-intern-dataset-and-lane-guide.md`, sections 8–9),
using the `framing-ml-problems` and `flyrank/flyrank-data` skills. A lane is provisional until
the end of Week 4 — the point here is an honest first frame, not a locked-in commitment.

## 1. My lane (or freestyle) and why

I picked **Lane 2 — Refresh / Content Opportunity Scoring**. The question it answers is the
clearest real decision in this internship data: *which pages should a content team review
first for refresh, expansion, protection, or pruning?*

Why this one, over the others:

- **The decision has a named human and a named action.** A content editor with a fixed number
  of review slots acts directly on the output (a ranked queue). No other lane maps as cleanly
  onto "limited capacity + thousands of candidates".
- **The starter pipeline already walks this lane end to end** (baseline rule → model → ranked
  queue with reason codes), so I have a transparent reference to beat instead of a blank page.
- **The starter label is weak on purpose** — `is_declining_label` is a current-window bucket,
  not a future outcome. The warehouse daily facts let me upgrade it to a stronger
  future-window label (features from the prior 90 days → decline/recovery over the next 30
  days) with a strict leakage audit. That upgrade is the capstone's main lift.
- **The data is deep here.** GSC impressions/clicks/position plus GA4 sessions/engagement
  exist for the bulk of rows (28.97M GSC-impression rows and 2.78M GA4-session rows in the
  warehouse), so both the traffic side and the engagement side of "refresh" are measurable.

Backup choices if the evidence later pushes me elsewhere: Lane 4 (CTR/Engagement Opportunity
Scoring) shares almost all the same plumbing, and the freeform *Growth / Recovery / Momentum
Prediction* direction is Lane 2 with a future-window label — a natural pivot if the data says
decline prediction beats refresh scoring. I can confirm or change all of this until Week 4.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
CSV = os.path.join("data", "raw", "content_refresh_anonymized.csv")
while not os.path.exists(os.path.join(ROOT, CSV)):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError(
            f"{CSV} not found from {os.getcwd()} — open this notebook from inside the Assignment-1 repo"
        )
    ROOT = parent

df = pd.read_csv(os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv"))
print(f"{len(df):,} content pages across {df['client_id'].nunique()} pseudonymized clients")

per_client = df.groupby("client_id")["content_id"].count()
print(f"pages per client — min {per_client.min():.0f} / median {per_client.median():.0f} / max {per_client.max():.0f}")
print("-> the review workload is far larger than any human capacity; a ranked queue is the product.")

## 2. The question: decision, action, cost of a wrong call

**What decision does this improve?** *Which page should a content editor review first?* The
output is not "predict decline" — it is a ranked review queue: pages ordered by how strongly
the evidence says they need attention, each carrying a suggested action (refresh / expand /
protect / prune / monitor) and a reason code a human can inspect.

**Who acts on the output, and what do they do?** A content editor or SEO reviewer at each
client, with a fixed weekly review budget (e.g. the top 20 or top 50 pages). They take the
top-K queue, verify each page, and decide what to edit. The queue only decides *order* — the
human still makes the call.

**What does a wrong answer cost?** Two-sided.

- A **false positive** wastes editor hours on a page that would not have moved — expensive,
  but recoverable.
- A **false negative** is worse: a high-visibility page keeps losing impressions while the
  team edits pages that matter less, so real traffic is lost silently.

That asymmetry is why the success metric should match how the list is used — **precision@K**
for a fixed review budget (the committed starter report shows a hand-written rule gets ~12 of
its top 50 right vs ~37 for a random forest: `outputs/model_report.md`, Precision@50
0.240 → 0.740) — not generic accuracy.

**Why does data or ML help at all?** Because the pattern is real but too messy to hand-write:
dozens of tangled signals (position, impressions, CTR, age, freshness, engagement, keyword
context) that shift over time, over thousands of pages per client. A simple rule misses most
of the queue; the workload below is too big to review by hand.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
CSV = os.path.join("data", "raw", "content_refresh_anonymized.csv")
while not os.path.exists(os.path.join(ROOT, CSV)):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError(
            f"{CSV} not found from {os.getcwd()} — open this notebook from inside the Assignment-1 repo"
        )
    ROOT = parent

df = pd.read_csv(os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv"))

visible = (df["impressions_90d"] >= 500).sum()
declining = (df["trend_direction"] == "down").sum()
print(f"visible pages (impressions_90d >= 500): {visible:,}")
print(f"pages in a declining trend: {declining:,}")
print("-> thousands of candidates for a handful of review slots: ranking by evidence, not a flat checklist.")

## 3. Quick look at the data (2-3 real numbers)

Every number below is **computed live** from `data/raw/content_refresh_anonymized.csv` —
nothing hardcoded. Three numbers that say this lane is worth the next 7 weeks:

1. **16,262 of 30,000 pages (54.2%) are currently in a declining trend.** Decline is not an
   edge case, it is the majority of the inventory — the review problem is big enough to
   matter.
2. **9,961 pages are both visible (impressions_90d ≥ 500) and declining — 59.6% of all
   visible pages.** This is the population where decline actually costs traffic, and it is
   far larger than any editorial capacity.
3. **61.3% of declining pages are visible.** Most of the decline problem sits on pages with
   real exposure — so *which* pages you pick first is the whole game, not a corner case.

The starter label here is a current-window proxy (`trend_direction == "down"`); the capstone
will re-derive it as a future-window outcome on the warehouse daily facts (prior 90 days →
next 30 days) with a leakage audit. The size of the opportunity is already visible in this
slice.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
CSV = os.path.join("data", "raw", "content_refresh_anonymized.csv")
while not os.path.exists(os.path.join(ROOT, CSV)):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError(
            f"{CSV} not found from {os.getcwd()} — open this notebook from inside the Assignment-1 repo"
        )
    ROOT = parent

df = pd.read_csv(os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv"))

n = len(df)
declining = (df["trend_direction"] == "down").sum()
visible = df["impressions_90d"] >= 500
declining_visible = visible & (df["trend_direction"] == "down")

print(f"1. Declining pages: {declining:,} of {n:,}  ({declining / n * 100:.1f}%)")
print(f"2. Visible AND declining: {int(declining_visible.sum()):,}  "
      f"({declining_visible.sum() / visible.sum() * 100:.1f}% of {int(visible.sum()):,} visible pages)")
print(f"3. Declining pages that are visible: {declining_visible.sum() / declining * 100:.1f}%")

## 4. Careful words: what I can and can't claim

**What this work will be able to say (observed / directional / decision-support):**

- Observed: on this slice, the majority of pages are currently in a declining trend, and
  most of that decline sits on visible pages.
- Directional: a ranking built from observable signals can surface the highest-attention
  pages first, and it will be measured honestly against a transparent baseline.
- Decision-support: the output is a *review queue for a human* — it ranks candidates, it
  does not promise that editing a page will recover its traffic.

**What it will never claim:**

- That a refresh *caused* a recovery — that needs an experiment or causal design, which this
  observational data cannot provide.
- That it proved a Google algorithm factor, an AI ranking, or an AI citation.
- Anything client-identifying: no client names, domains, URLs, titles, or raw queries in any
  output or chart.

In [ ]:
claims = {
    "safe to claim (observed / directional / decision-support)": [
        "observed: most pages in this slice are in a declining trend",
        "observed: most declining pages have real visibility, so order matters",
        "directional: a learned ranking beat the rule on the starter slice (Precision@50 0.740 vs 0.240)",
        "decision-support: output is a ranked review queue for a human editor",
    ],
    "never claimed": [
        "a refresh caused a recovery (no causal design)",
        "proved a Google algorithm factor, AI ranking, or citation",
        "anything that identifies a client, URL, title, or raw query",
    ],
}
for heading, lines in claims.items():
    print(heading + ":")
    for line in lines:
        print("  -", line)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.